# Document OCR Pipeline — Complete Colab Walkthrough

*Part 1 of 4 · From pixels to structured JSON*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/01_document_ocr_pipeline.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/01_document_ocr_pipeline.ipynb

---

## What this notebook is about

This is a **hands-on blog-style walkthrough** of the full document OCR pipeline implemented in `vlm_pipeline/backends/florence2.py` (with a Qwen equivalent). We do not skip steps: every public function in the backend is exercised in order, with the math behind each stage spelled out.

At a high level, a vision–language OCR model solves:

$$
P(\mathbf{y} \mid \mathbf{I}, t) = \prod_{k=1}^{K} P(y_k \mid y_{<k}, \mathbf{I}, t)
$$

where $\mathbf{I}$ is the document image, $t$ is a **task prompt** (e.g. `<OD>` for object detection), and $\mathbf{y} = (y_1,\ldots,y_K)$ is the autoregressively generated token sequence that encodes boxes, text, layout, or tables.

**Proof (chain rule decomposition):** by autoregressive factorization, $\log P(\mathbf{y}\mid\mathbf{I},t) = \sum_{k=1}^K \log P(y_k\mid y_{<k}, \mathbf{I}, t)$ — each factor is one forward pass of $f_\theta$; total log-prob is sum of per-step cross-entropies.

| Stage | Functions covered |
|-------|-------------------|
| **1** | `resolve_device`, `AutoProcessor`, tokenizer, `AutoModelForCausalLM`, `model.eval()` |
| **2** | `pad_info`, `processor()`, `model.generate()` |
| **3** | `batch_decode()` → raw text |
| **4** | `_parse_florence`, `post_process_generation`, `clean_florence_text` |
| **5** | `quad_to_bbox`, `unmap_bbox`, `_detect/_ocr/_layout/_table`, `task_result()` |
| **→2** | Preview **SmoothQuant**, **SpinQuant**, **ConvRot** → full quant in notebook 02 |

**Series:** [01 OCR](01_document_ocr_pipeline.ipynb) → [02 Quantization (GPTQ · AWQ · SmoothQuant · SpinQuant · ConvRot)](02_ocr_pipeline_quant.ipynb) → [03 Mobile export](03_ocr_pipeline_mobile.ipynb) → [04 Mobile complete](04_ocr_pipeline_mobile_complete.ipynb)

Set **`BACKEND`** and **`TASK`** in the config cell below.  
**GPU runtime** recommended (Runtime → Change runtime type → T4 GPU).

## 0 — Install & config

Before we load weights, we pin the software stack. Florence-2 requires **`transformers==4.49.x`**; Qwen-VL needs **`>=4.51`**. Mismatch causes silent tokenizer or architecture errors.

**Steps:**

1. Set **`BACKEND`** in the first code cell (`"florence2"` or `"qwen"`).
2. Run that cell — it installs deps and **auto-restarts** the runtime if the wrong `transformers` was loaded.
3. After a restart, click **Run all** (the first cell runs again; Stage 1 should then show `transformers 4.49.x` for Florence-2).

### Why version pinning matters (formal view)

Let $\mathcal{V}_v$ be the vocabulary and $\mathcal{P}_v$ the post-processor for transformers version $v$. Inference requires:

$$
\mathcal{P}_v \circ \mathcal{D}_v \circ \mathcal{G}_\theta = \text{JSON} \quad \text{only if } v = v^\star
$$

where $\mathcal{D}_v$ decodes token IDs and $v^\star = 4.49$ for Florence-2. If $v \neq v^\star$, token IDs may still decode to *some* string, but $\mathcal{P}_v$ is applied with the wrong grammar → parse failure. **Proof:** parsers are hard-coded to delimiter tokens present in $\mathcal{V}_{v^\star}$; a shifted vocabulary breaks bijection between generated strings and coordinate fields.

In [ ]:
import os
import re
import subprocess
import sys

# ── CONFIG (set backend here — install uses this) ────────

# ── Backend & Task ──
BACKEND = "florence2"       # "florence2" (fast) | "qwen" (best quality)
TASK = "detect"             # detect | ocr | layout | table
RUN_ALL_TASKS = False       # True = run all 4 tasks sequentially

# ── Generation ──
MAX_NEW_TOKENS = 2048       # max output tokens per task

def _pip_version(package):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        capture_output=True, text=True, check=False,
    )
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""

def _restart_runtime():
    print("Restarting runtime so the correct transformers version loads...")
    os.kill(os.getpid(), 9)

def ensure_transformers(backend):
    """Install the right transformers and restart if an old version is still in memory."""
    if backend == "florence2":
        target = "4.49.0"
        ok = lambda v: v.startswith("4.49")
    else:
        target = ">=4.51,<5.0"
        ok = lambda v: tuple(int(x) for x in v.split(".")[:2]) >= (4, 51)

    pip_ver = _pip_version("transformers")
    if not ok(pip_ver):
        print(f"Installing transformers {target} (pip had {pip_ver or 'none'})...")
        if backend == "florence2":
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "transformers==4.49.0",
            ])
        else:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--upgrade", "transformers>=4.51,<5.0",
            ])
        pip_ver = _pip_version("transformers")
        if not ok(pip_ver):
            raise RuntimeError(f"Could not install transformers {target}. pip reports {pip_ver}")

    try:
        import transformers
        loaded = transformers.__version__
    except ImportError:
        loaded = None

    if loaded and not ok(loaded):
        print(f"pip has {pip_ver} but Python still has {loaded} loaded. Restarting once...")
        _restart_runtime()

    return pip_ver

# ── Install other dependencies ─────────────────────────────
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "scipy>=1.12", "scikit-learn",
    "torch", "accelerate", "pillow", "qwen-vl-utils",
    "huggingface_hub", "bitsandbytes", "matplotlib", "requests",
])

_tf_ver = ensure_transformers(BACKEND)
print(f"BACKEND={BACKEND}  |  transformers {_tf_ver}")
print("If the runtime restarted, click Run all from the top.")

In [ ]:
import json
import re
import tempfile
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont

# BACKEND, TASK, RUN_ALL_TASKS, MAX_NEW_TOKENS are set in the cell above
MODEL_IDS = {                           # map backend name → HuggingFace model ID
    "florence2": "microsoft/Florence-2-base-ft",
    "qwen": "Qwen/Qwen2.5-VL-3B-Instruct",
}
FLORENCE_PROMPTS = {                     # Florence-2 uses special task tokens as prompts
    "detect": "<OCR_WITH_REGION>",      # bounding boxes + text
    "ocr": "<OCR>",                     # plain text extraction
    "layout": "<DENSE_REGION_CAPTION>", # region descriptions
    "table": "<OCR>",                   # table cells as text
}
QWEN_PROMPTS = {
    "layout": """Analyze this document page layout.
Return ONLY valid JSON (no markdown fences):
{"blocks": [{"label": "Title|Text|Table|Figure|List|Header|Footer", "bbox": [x1,y1,x2,y2], "reading_order": 1}]}
Use pixel coordinates from the image. Preserve reading order top-to-bottom.""",
    "ocr": """OCR this document page.
Return clean reading-order text as HTML using <p> tags.
Preserve Hindi and English exactly as written. Do not summarize.""",
    "table": """Find all tables on this page.
Return ONLY valid JSON (no markdown fences):
{"tables": [{"bbox": [x1,y1,x2,y2], "rows": [["cell text"]]}]}
If no table exists, return {"tables": []}.""",
    "detect": """Read all visible text lines on this page.
Return ONLY valid JSON (no markdown fences):
{"text_lines": [{"text": "line text", "bbox": [x1,y1,x2,y2], "confidence": 0.95}]}
Use pixel coordinates.""",
}
HANDLERS_MAP = {
    "detect": "_detect",
    "ocr": "_ocr",
    "layout": "_layout",
    "table": "_table",
}

MODEL_ID = MODEL_IDS[BACKEND]
PROMPT = FLORENCE_PROMPTS[TASK] if BACKEND == "florence2" else QWEN_PROMPTS[TASK]

def resolve_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def florence_model_dtype(model):
    return next(model.parameters()).dtype

def prepare_florence_inputs(processor, prompt, padded, device, model):
    inputs = processor(text=prompt, images=padded, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype=florence_model_dtype(model))
    return inputs

DEVICE = resolve_device()
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Backend: {BACKEND}  |  Task: {TASK}  |  Model: {MODEL_ID}")
print(f"Handler: {HANDLERS_MAP[TASK]}")

---
## Stage 1 — `__init__`: load processor, tokenizer, model

Mirrors `Florence2Backend.__init__` in `florence2.py`.

We assemble three objects that together implement the conditional distribution $P(\mathbf{y}\mid\mathbf{I},t)$:

1. **`resolve_device()`** — pick CPU or CUDA (inference is $O(K \cdot L)$ in generated tokens $K$ and model depth $L$; GPU amortizes matmul cost).
2. **`AutoProcessor.from_pretrained()`** — tokenizer + image preprocessor. Maps text $t$ to token IDs and image $\mathbf{I}$ to pixel tensor $\mathbf{X} \in \mathbb{R}^{C \times H \times W}$.
3. **`AutoModelForCausalLM.from_pretrained()`** — loads $\theta$ (weights) for the vision encoder + decoder stack.
4. **`model.eval()`** — disables dropout and batch-norm training noise; fixes the forward map $f_\theta$ for deterministic inference.

**Proof sketch (eval mode):** in training, dropout masks activations randomly; at inference we want $\mathbb{E}[f_\theta(\mathbf{x})]$ under the trained weights, not a stochastic sample. `eval()` plus `torch.no_grad()` ensures $\nabla_\theta \mathcal{L} = 0$ and no gradient buffers are allocated.

In [ ]:
ensure_transformers(BACKEND)  # install correct transformers version for chosen backend
import transformers

print("Step 1.1 — resolve_device()")
print(f"  → {DEVICE}")
print(f"  transformers {transformers.__version__}")

print("\nStep 1.2 — AutoProcessor.from_pretrained()")
if BACKEND == "florence2":
    from transformers import AutoProcessor, AutoModelForCausalLM
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)  # loads tokenizer + image processor
else:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(MODEL_ID)

tok = processor.tokenizer
print(f"  Processor class : {type(processor).__name__}")
print(f"  Tokenizer class : {type(tok).__name__}")
print(f"  Vocab size      : {tok.vocab_size}")
print(f"  Special tokens  : {tok.special_tokens_map}")

In [ ]:
print("Step 1.3 — Tokenizer demo (prompt → token IDs → decoded)")
sample = PROMPT if BACKEND == "florence2" else PROMPT[:80]
ids = tok.encode(sample)
print(f"  Prompt   : {sample[:100]}{'...' if len(sample)>100 else ''}")
print(f"  Token IDs: {ids[:12]}{'...' if len(ids)>12 else ''}")
print(f"  Count    : {len(ids)} tokens")
print(f"  Decoded  : {tok.decode(ids[:20])}")

In [ ]:
print("Step 1.4 — AutoModelForCausalLM.from_pretrained() + model.eval()")
if BACKEND == "florence2":
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32  # fp16 on GPU for speed
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,       # Florence-2 has custom code in repo
        torch_dtype=dtype,
        attn_implementation="eager",  # avoid flash-attn issues on older GPUs
    ).to(DEVICE)
else:
    kwargs = {}
    if DEVICE == "cuda":
        from transformers import BitsAndBytesConfig
        kwargs = {
            "device_map": "auto",
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16
            ),
        }
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)
    if DEVICE != "cuda":
        model = model.to(DEVICE)

model.eval()
params = sum(p.numel() for p in model.parameters())
print(f"  Model class : {type(model).__name__}")
print(f"  Parameters  : {params/1e6:.1f} M")
print(f"  Training?   : {model.training}  (False = eval mode ✓)")
print(f"  Device      : {next(model.parameters()).device}")

cfg = model.config
for k in ["model_type", "hidden_size", "num_hidden_layers", "vocab_size"]:
    if hasattr(cfg, k):
        print(f"  config.{k}: {getattr(cfg, k)}")

---
## Helper functions (from `vlm_pipeline/utils/`)

These utilities encode **geometry** (padding, bbox unmapping) and **text cleanup**. All coordinate math is a single affine chain.

### Padding as translation

Define forward map from original pixel $(x, y)$ to padded canvas $(x_p, y_p)$:

$$
\begin{pmatrix} x_p \\ y_p \end{pmatrix} = \begin{pmatrix} x \\ y \end{pmatrix} + \begin{pmatrix} \text{pad}_x \\ \text{pad}_y \end{pmatrix} = \mathbf{T}\begin{pmatrix} x \\ y \\ 1 \end{pmatrix}, \quad \mathbf{T} = \begin{pmatrix} 1 & 0 & \text{pad}_x \\ 0 & 1 & \text{pad}_y \\ 0 & 0 & 1 \end{pmatrix}
$$

**Proof (inverse exists):** $\mathbf{T}$ is translation-only with $\det(\mathbf{T}) = 1$, so $\mathbf{T}^{-1}$ exists and subtracts $(\text{pad}_x, \text{pad}_y)$. No scaling ⇒ aspect ratio preserved inside content region.

### `quad_to_bbox`

For quadrilateral vertices $\{(x_i,y_i)\}_{i=1}^4$:

$$
x_{\min} = \min_i x_i,\; x_{\max} = \max_i x_i,\; y_{\min} = \min_i y_i,\; y_{\max} = \max_i y_i
$$

Axis-aligned bbox is the **smallest enclosing rectangle** (tightest AABB). `unmap_bbox` applies $\mathbf{T}^{-1}$ to all four corners.

In [ ]:
def pad_info(image):
    """Pad image to square — Florence-2 expects square inputs. Returns offsets for unmap."""
    w, h = image.size
    side = max(w, h)                                 # target square side length
    canvas = Image.new("RGB", (side, side), "white") # white-padded square
    pad_x, pad_y = (side - w) // 2, (side - h) // 2 # centering offsets
    canvas.paste(image, (pad_x, pad_y))              # paste original centered
    return canvas, pad_x, pad_y, w, h

def clean_html_tags(text):
    """Strip any XML/HTML tags from model output (Florence sometimes emits these)."""
    return re.sub(r"</?[a-zA-Z_][^>]*>", "", text).strip()

def clean_florence_text(text):
    """Clean Florence-2 raw output text."""
    return clean_html_tags(text)

def clean_model_response(text):
    text = text.strip()
    if "assistant" in text:
        text = text.split("assistant", 1)[-1].strip()
    fence = re.search(r"```(?:html|json)?\s*([\s\S]*?)\s*```", text)
    return fence.group(1).strip() if fence else text

def extract_json(text):
    text = clean_model_response(text)
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if fence:
        text = fence.group(1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*\}", text)
        if match:
            return json.loads(match.group(0))
        raise

def decode_qwen_output(processor, inputs, generated_ids):
    """Decode only newly generated tokens (skip prompt/image tokens)."""
    input_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_len:]
    return processor.batch_decode(
        new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

def scale_bbox_to_image(bbox, width, height):
    if not bbox or len(bbox) != 4:
        return bbox
    x1, y1, x2, y2 = (float(v) for v in bbox)
    if max(x1, y1, x2, y2) <= 1.0:
        x1, x2 = x1 * width, x2 * width
        y1, y2 = y1 * height, y2 * height
    elif max(x1, x2) > width or max(y1, y2) > height:
        x1, x2 = x1 * width / 1000, x2 * width / 1000
        y1, y2 = y1 * height / 1000, y2 * height / 1000
    return [int(x1), int(y1), int(x2), int(y2)]

def scale_qwen_result_bboxes(result, width, height):
    result = dict(result)
    for line in result.get("text_lines", []):
        line["bbox"] = scale_bbox_to_image(line.get("bbox"), width, height)
    for block in result.get("blocks", []):
        block["bbox"] = scale_bbox_to_image(block.get("bbox"), width, height)
    for table in result.get("tables", []):
        table["bbox"] = scale_bbox_to_image(table.get("bbox"), width, height)
    return result

def run_qwen_generate(image, processor, model, prompt, max_new_tokens):
    from qwen_vl_utils import process_vision_info
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        temp_path = tmp.name
    image.save(temp_path)
    try:
        messages = [{"role": "user", "content": [
            {"type": "image", "image": temp_path},
            {"type": "text", "text": prompt},
        ]}]
        chat_text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[chat_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
        raw_text = decode_qwen_output(processor, inputs, generated_ids)
        return raw_text, inputs, generated_ids
    finally:
        Path(temp_path).unlink(missing_ok=True)

def run_qwen_task(task, image, processor, model, max_tokens=2048):
    prompt = QWEN_PROMPTS[task]
    raw_text = run_qwen_generate(image, processor, model, prompt, max_tokens)[0]
    cleaned = clean_model_response(raw_text)
    if task == "ocr":
        return task_result("ocr", "qwen", text=cleaned, html=cleaned)
    try:
        parsed = extract_json(cleaned)
    except json.JSONDecodeError:
        parsed = {"raw": cleaned}
    result = task_result(task, "qwen")
    result.update(parsed if isinstance(parsed, dict) else {"raw": parsed})
    w, h = image.size
    return scale_qwen_result_bboxes(result, w, h)

def quad_to_bbox(quad):
    xs, ys = quad[0::2], quad[1::2]
    return [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]

def unmap_bbox(bbox, pad_x, pad_y, orig_w, orig_h):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(orig_w, x1 - pad_x)); y1 = max(0, min(orig_h, y1 - pad_y))
    x2 = max(0, min(orig_w, x2 - pad_x)); y2 = max(0, min(orig_h, y2 - pad_y))
    return [] if x2 <= x1 or y2 <= y1 else [x1, y1, x2, y2]

def task_result(task, backend, **payload):
    return {"task": task, "backend": backend, **payload}

def parse_florence(processor, task_prompt, generated_text, pad_x, pad_y, orig_w, orig_h):
    """_parse_florence() — calls post_process_generation."""
    side = max(orig_w, orig_h)
    parsed = processor.post_process_generation(
        generated_text, task=task_prompt, image_size=(side, side)
    )
    return parsed, pad_x, pad_y, orig_w, orig_h

print("Helpers loaded: pad_info, decode_qwen_output, run_qwen_task, parse_florence, task_result")

---
## Load document image

Upload your own file or use the bundled sample page.

The pipeline expects an RGB image $\mathbf{I} \in [0,255]^{H_0 \times W_0 \times 3}$. All later bbox math refers back to $(W_0, H_0)$ — the **original** dimensions before square padding.

In [ ]:
# Option A — upload (uncomment in Colab)
# from google.colab import files
# uploaded = files.upload()
# image = Image.open(list(uploaded.keys())[0]).convert("RGB")

# Option B — sample from repo
try:
    url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
    image = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
except Exception:
    image = Image.new("RGB", (640, 480), "white")
    ImageDraw.Draw(image).text((20, 20), "Sample document", fill="black")

print(f"Image size: {image.size[0]} x {image.size[1]} px")
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.title("Input document page"); plt.axis("off"); plt.show()

---
## Stage 2 — `_generate()` steps A→C: preprocess + inference

This stage implements the **encoder input** and **autoregressive decode** loop.

| Step | Function | Output |
|------|----------|--------|
| **A** | `pad_info(image)` | square image + `pad_x, pad_y, orig_w, orig_h` |
| **B** | `processor(text, images)` | `input_ids`, `pixel_values` tensors |
| **C** | `model.generate(...)` | output **token ID** tensor |

### Step A — Square padding (geometry)

Let $W_0, H_0$ be original width and height. We pad to side $S = \max(W_0, H_0)$:

$$
\text{pad}_x = \left\lfloor \frac{S - W_0}{2} \right\rfloor, \quad
\text{pad}_y = \left\lfloor \frac{S - H_0}{2} \right\rfloor
$$

**Proof (no anisotropic stretch):** scale factor $s_x = s_y = 1$ on content pixels; only translation added. For any two points $(x_a,y_a), (x_b,y_b)$ in the document:

$$
\frac{x_b - x_a}{y_b - y_a} \text{ unchanged after pad } \Rightarrow \text{angles and aspect ratio preserved}
$$

### Step B — Tokenization

Task prompt $t$ becomes $\mathbf{x}^{\text{text}} = \text{tokenize}(t)$. Images become $\mathbf{X} \in \mathbb{R}^{1 \times C \times H \times W}$ after normalization $\mathbf{X} = (\mathbf{I}/255 - \mu)/\sigma$.

### Step C — Autoregressive generation

At step $k$, logits $\mathbf{z}_k \in \mathbb{R}^{|\mathcal{V}|}$ and (greedy) choice:

$$
P(y_k = v \mid y_{<k}, \mathbf{X}, t) = \frac{e^{z_{k,v}}}{\sum_{v'} e^{z_{k,v'}}}, \quad y_k = \arg\max_v P(y_k = v \mid \cdot)
$$

**Complexity:** $O(K \cdot C_{\text{layer}})$ where $C_{\text{layer}} = O(L d^2)$ for $L$ layers, hidden $d$, and $K$ new tokens.

In [ ]:
pad_meta = {}

if BACKEND == "florence2":
    print("Step 2A — pad_info(image)")
    padded, pad_x, pad_y, orig_w, orig_h = pad_info(image)
    pad_meta = dict(pad_x=pad_x, pad_y=pad_y, orig_w=orig_w, orig_h=orig_h)
    print(f"  Original : {orig_w} x {orig_h}")
    print(f"  Padded   : {padded.size[0]} x {padded.size[1]} (square)")
    print(f"  pad_x={pad_x}, pad_y={pad_y}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].imshow(image); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(padded); ax[1].set_title(f"Padded square ({padded.size[0]}px)"); ax[1].axis("off")
    rect = patches.Rectangle((pad_x, pad_y), orig_w, orig_h, linewidth=2, edgecolor="red", facecolor="none")
    ax[1].add_patch(rect); ax[1].set_title("Padded (red = original area)"); plt.show()

    print(f"\nStep 2B — processor(text='{PROMPT}', images=padded)")
    inputs = prepare_florence_inputs(processor, PROMPT, padded, DEVICE, model)
    print(f"  input_ids     shape: {tuple(inputs['input_ids'].shape)}")
    print(f"  pixel_values  shape: {tuple(inputs['pixel_values'].shape)}")
    print(f"  pixel_values  dtype: {inputs['pixel_values'].dtype} (model: {florence_model_dtype(model)})")
    print(f"  input_ids[0][:15]: {inputs['input_ids'][0][:15].tolist()}")

    print(f"\nStep 2C — model.generate(max_new_tokens={MAX_NEW_TOKENS}, num_beams=1)")
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=1,
            use_cache=False,
        )
else:
    print("Step 2 — Qwen: apply_chat_template + process_vision_info + generate")
    raw_text, inputs, generated_ids = run_qwen_generate(
        image, processor, model, PROMPT, MAX_NEW_TOKENS
    )
    print(f"  input_ids shape: {tuple(inputs['input_ids'].shape)}")
    print(f"  response preview: {raw_text[:200]}...")

print(f"\n  generated_ids shape : {tuple(generated_ids.shape)}")
print(f"  Total tokens        : {generated_ids.shape[1]}")
print(f"  First 20 token IDs  : {generated_ids[0][:20].tolist()}")
print(f"  Last 10 token IDs   : {generated_ids[0][-10:].tolist()}")

---
## Stage 3 — `_generate()` step D: `batch_decode()` → raw text

Token IDs $\mathbf{y} = (y_1,\ldots,y_K)$ decode via vocabulary $\mathcal{V}: \{1,\ldots,|\mathcal{V}|\} \to \Sigma$:

$$
\text{raw\_text} = \mathcal{D}(\mathbf{y}) = \mathcal{V}[y_1] \circ \mathcal{V}[y_2] \circ \cdots \circ \mathcal{V}[y_K]
$$

where $\circ$ is string concatenation.

### Why `skip_special_tokens=False`

Structure tokens $\mathcal{S} \subset \Sigma$ (location markers, delimiters) are required for parsing:

$$
\mathcal{G}_\tau \circ \mathcal{D}(\mathbf{y}) = \emptyset \text{ if } \mathcal{S} \cap \text{tokens}(\mathcal{D}(\mathbf{y})) = \emptyset
$$

**Proof:** $\mathcal{G}_\tau$ is a grammar over alphabet including $\mathcal{S}$; removing specials before parse is equivalent to deleting punctuation from JSON — parser cannot recover structure.

OCR output is a **structured token language**, not plain prose.

In [ ]:
print("Step 3 — decode model output")
if BACKEND == "florence2":
    raw_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    raw_clean = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"  skip_special_tokens=False → {len(raw_text)} chars (used for parsing)")
    print(f"  skip_special_tokens=True  → {len(raw_clean)} chars (human readable)")
else:
    raw_text = decode_qwen_output(processor, inputs, generated_ids)
    raw_clean = clean_model_response(raw_text)
    print(f"  new tokens only → {len(raw_text)} chars")
    print(f"  after clean_model_response → {len(raw_clean)} chars")

print("\n" + "="*70)
print("RAW TEXT (full model output)")
print("="*70)
print(raw_text[:2500])
if len(raw_text) > 2500:
    print(f"\n... [{len(raw_text)-2500} more characters]")

In [ ]:
fig = plt.figure(figsize=(15, 9))
ax1 = fig.add_subplot(2, 2, 1); ax1.imshow(image); ax1.set_title("① Input image"); ax1.axis("off")
ax2 = fig.add_subplot(2, 2, 2); ax2.axis("off")
ax2.text(0, 1, f"② Task: {TASK}\nBackend: {BACKEND}\n\nPrompt:\n{PROMPT[:400]}", va="top", fontsize=9, family="monospace")
ax2.set_title("Prompt sent to model")
ax3 = fig.add_subplot(2, 1, 2); ax3.axis("off")
ax3.text(0, 1, f"③ Raw decoded text ({len(raw_text)} chars):\n\n{raw_text[:1800]}", va="top", fontsize=8, family="monospace")
plt.tight_layout(); plt.show()

---
## Stage 4 — `_parse_florence` + `post_process_generation` + clean

The raw string is parsed into typed Python structures (boxes, labels, text blocks):

```python
parsed = processor.post_process_generation(
    generated_text,
    task=task_prompt,
    image_size=(max(orig_w, orig_h), max(orig_w, orig_h)),
)
```

### Parser as a grammar

Let $\mathcal{G}_t = (N, \Sigma, P, S)$ be a context-free grammar where $\Sigma$ includes location tokens. Parsing:

$$
\text{parsed} = \mathcal{G}_t\bigl(\text{raw\_text},\ (S, S)\bigr), \quad S = \max(W_0, H_0)
$$

Coordinates in the parse tree live in **padded-square space** $[0,S]^2$. **Correctness condition:** `image_size` passed to the parser must equal the $S$ used during generation — otherwise normalized coords scale incorrectly:

$$
x_{\text{orig}} = x_{\text{parsed}} \cdot \frac{W_0}{S} \text{ (wrong if } S_{\text{parse}} \neq S_{\text{gen}} \text{)}
$$

`clean_florence_text` applies idempotent string normalization $c \circ c = c$ (whitespace collapse, artifact removal) without altering semantic tokens.

In [ ]:
print("Step 4 — _parse_florence / clean_model_response")

if BACKEND == "florence2":
    parsed, px, py, ow, oh = parse_florence(
        processor, PROMPT, raw_text,
        pad_meta["pad_x"], pad_meta["pad_y"],
        pad_meta["orig_w"], pad_meta["orig_h"],
    )
    region = parsed.get(PROMPT, {})
    print(f"  post_process_generation keys: {list(parsed.keys())}")
    if isinstance(region, dict):
        print(f"  Region keys: {list(region.keys())}")
        for k, v in region.items():
            if isinstance(v, list):
                print(f"    {k}: {len(v)} items  (first: {v[0] if v else 'empty'})")
            else:
                print(f"    {k}: {str(v)[:80]}")
    else:
        print(f"  Plain text region: {str(region)[:200]}")
else:
    cleaned = raw_clean
    print(f"  clean_model_response: {len(cleaned)} chars")
    print(f"  Preview: {cleaned[:600]}")
    try:
        parsed = extract_json(cleaned)
        print(f"  extract_json keys: {list(parsed.keys()) if isinstance(parsed, dict) else type(parsed)}")
    except json.JSONDecodeError as e:
        parsed = {"raw": cleaned}
        print(f"  JSON parse failed: {e}")

In [ ]:
print("BEFORE vs AFTER cleaning")
print("-"*50)
print("BEFORE (raw_text):")
print(raw_text[:350])
print("\n" + "-"*50)
if BACKEND == "florence2":
    region = parsed.get(PROMPT, {})
    if isinstance(region, dict) and region.get("labels"):
        print("AFTER (clean_florence_text on first 3 labels):")
        for lbl in region["labels"][:3]:
            print(f"  raw: {lbl!r}  →  clean: {clean_florence_text(str(lbl))!r}")
    elif isinstance(region, str) or (isinstance(region, dict) is False):
        t = str(region if not isinstance(region, dict) else region)
        print(f"AFTER clean_florence_text: {clean_florence_text(t)[:300]}")
else:
    print(f"AFTER clean_model_response: {clean_model_response(raw_text)[:350]}")

---
## Stage 4b — `quad_to_bbox` + `unmap_bbox` demo (detect/layout)

Florence emits quadrilaterals $\mathbf{q} = (x_1,y_1,\ldots,x_4,y_4)$ on the $S \times S$ canvas.

### Step 1 — AABB from quad

$$
\text{bbox}(\mathbf{q}) = [x_{\min}, y_{\min}, x_{\max}, y_{\max}], \quad x_{\min} = \min_i x_i
$$

**Proof (minimal enclosing AABB):** any axis-aligned box containing all vertices must satisfy $x_{\min} \le x_i \le x_{\max}$ for all $i$; choosing $x_{\min} = \min_i x_i$ and $x_{\max} = \max_i x_i$ is tight.

### Step 2 — Unmap (inverse padding)

$$
\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} x \\ y \end{pmatrix} - \begin{pmatrix} \text{pad}_x \\ \text{pad}_y \end{pmatrix}
$$

**Proof:** forward pad was $\mathbf{T}$ with $\det(\mathbf{T})=1$; inverse is unique translation. Valid document coords require $0 \le x' < W_0$, $0 \le y' < H_0$; otherwise the point lies in letterbox margin → clip or discard.

In [ ]:
if BACKEND == "florence2" and TASK in ("detect", "layout"):
    region = parsed.get(PROMPT, {})
    px, py, ow, oh = pad_meta["pad_x"], pad_meta["pad_y"], pad_meta["orig_w"], pad_meta["orig_h"]

    if TASK == "detect" and region.get("quad_boxes"):
        quad = region["quad_boxes"][0]
        label = region["labels"][0]
        bbox_padded = quad_to_bbox(quad)
        bbox_orig = unmap_bbox(bbox_padded, px, py, ow, oh)
        print(f"Demo line 1: {clean_florence_text(str(label))}")
        print(f"  quad (8 pts)     : {quad}")
        print(f"  quad_to_bbox     : {bbox_padded}  (on padded image)")
        print(f"  unmap_bbox       : {bbox_orig}  (on original image)")

        vis = image.copy(); draw = ImageDraw.Draw(vis)
        draw.rectangle(bbox_orig, outline="lime", width=3)
        draw.text((bbox_orig[0], max(0,bbox_orig[1]-14)), clean_florence_text(str(label))[:30], fill="lime")
        plt.figure(figsize=(8,6)); plt.imshow(vis); plt.title("First detected line — bbox on original image"); plt.axis("off"); plt.show()
    elif TASK == "layout" and region.get("bboxes"):
        bbox = [int(v) for v in region["bboxes"][0]]
        mapped = unmap_bbox(bbox, px, py, ow, oh)
        print(f"Demo block 1: {clean_florence_text(str(region['labels'][0]))[:60]}")
        print(f"  bbox on padded : {bbox}")
        print(f"  unmap_bbox     : {mapped}")
else:
    print("(quad_to_bbox / unmap_bbox demo skipped — set BACKEND=florence2, TASK=detect or layout)")

---
## Stage 5 — Task handlers → `task_result()` JSON

Mirrors `_dispatch`, `_detect`, `_ocr`, `_layout`, `_table` from `florence2.py`.

### Dispatch as a function table

Let $\mathcal{T} = \{\text{detect}, \text{ocr}, \text{layout}, \text{table}\}$. The entry point is:

$$
\text{task\_result}(\mathbf{I}, \tau) = \mathcal{H}_\tau\bigl(\mathcal{G}_\tau \circ \mathcal{D} \circ \mathcal{G}_\theta(\mathbf{I}, p_\tau)\bigr), \quad \tau \in \mathcal{T}
$$

where $p_\tau$ is the task prompt (`<OD>`, `<OCR>`, etc.) and $\mathcal{H}_\tau$ normalizes output schema.

| Task | Prompt $p_\tau$ | Output schema |
|------|-----------------|---------------|
| **detect** | object detection | boxes + labels |
| **ocr** | text regions | strings + optional boxes |
| **layout** | reading order | blocks + order index |
| **table** | grid structure | rows / cells |

**Proof (shared core):** all handlers call the same `_generate` — only $(p_\tau, \mathcal{G}_\tau, \mathcal{H}_\tau)$ differ. So bug fixes to padding or decode propagate to every task automatically.

In [ ]:
def run_florence_task(task, image, processor, model, device, max_tokens=2048):
    """Full Florence-2 pipeline for one task: pad → tokenize → generate → parse → result."""
    prompt = FLORENCE_PROMPTS[task]                    # task-specific special token
    tokens = 1024 if task == "layout" else max_tokens  # layout needs fewer tokens
    padded, px, py, ow, oh = pad_info(image)          # pad to square + save offsets
    inputs = prepare_florence_inputs(processor, prompt, padded, device, model)  # tokenize text+image
    with torch.no_grad():                             # inference only — no grad needed
        gen = model.generate(
            input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
            max_new_tokens=tokens, num_beams=1, use_cache=False,  # greedy decoding
        )
    raw = processor.batch_decode(gen, skip_special_tokens=False)[0]  # decode token IDs → text
    parsed, px, py, ow, oh = parse_florence(processor, prompt, raw, px, py, ow, oh)  # structure output

    if task == "detect":
        region = parsed.get(prompt, {})
        lines = []
        for quad, label in zip(region.get("quad_boxes", []), region.get("labels", [])):
            bbox = unmap_bbox(quad_to_bbox(quad), px, py, ow, oh)
            if bbox:
                lines.append({"text": clean_florence_text(label), "bbox": bbox, "confidence": 1.0})
        return task_result("detect", "florence2", text_lines=lines)
    if task == "ocr":
        text = parsed.get(prompt, "")
        text = clean_florence_text(str(text) if not isinstance(text, str) else text)
        return task_result("ocr", "florence2", text=text, html=f"<p>{text}</p>")
    if task == "layout":
        labels = parsed.get(prompt, {}).get("labels", [])
        bboxes = parsed.get(prompt, {}).get("bboxes", [])
        blocks = []
        for i, (bbox, label) in enumerate(zip(bboxes, labels), start=1):
            mapped = unmap_bbox([int(v) for v in bbox], px, py, ow, oh)
            if mapped:
                blocks.append({"label": "Text", "bbox": mapped, "reading_order": i,
                               "caption": clean_florence_text(str(label))})
        return task_result("layout", "florence2", blocks=blocks)
    ocr = run_florence_task("ocr", image, processor, model, device)
    return task_result("table", "florence2", tables=[], note="Use qwen for tables",
                       fallback_text=ocr.get("text", ""))

print(f"Running handler: {HANDLERS_MAP[TASK]}")
if BACKEND == "florence2":
    result = run_florence_task(TASK, image, processor, model, DEVICE, MAX_NEW_TOKENS)
else:
    result = run_qwen_task(TASK, image, processor, model, MAX_NEW_TOKENS)

print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
def draw_result(image, result, task):
    vis = image.copy(); draw = ImageDraw.Draw(vis)
    if task == "detect" and result.get("text_lines"):
        for line in result["text_lines"]:
            b = line.get("bbox")
            if not b or len(b) != 4:
                continue
            draw.rectangle(b, outline="red", width=2)
            draw.text((b[0], max(0, b[1]-12)), str(line.get("text", ""))[:32], fill="red")
        return vis, f"detect — {len(result['text_lines'])} lines", "red"
    if task == "layout" and result.get("blocks"):
        for block in result["blocks"]:
            b = block.get("bbox")
            if not b or len(b) != 4:
                continue
            draw.rectangle(b, outline="blue", width=2)
            draw.text((b[0], b[1]), f"#{block.get('reading_order', '?')}", fill="blue")
        return vis, f"layout — {len(result['blocks'])} blocks", "blue"
    if task == "table" and result.get("tables"):
        for i, table in enumerate(result["tables"], start=1):
            b = table.get("bbox")
            if not b or len(b) != 4:
                continue
            draw.rectangle(b, outline="green", width=2)
            draw.text((b[0], b[1]), f"table {i}", fill="green")
        return vis, f"table — {len(result['tables'])} tables", "green"
    note = result.get("raw", "")[:120] if result.get("raw") else "no boxes parsed"
    return vis, f"{task} — {note}", "black"

if TASK == "ocr":
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(image); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].axis("off")
    axes[1].text(0, 1, result.get("text", "")[:3000], va="top", fontsize=9, family="monospace")
    axes[1].set_title("OCR text output")
else:
    vis, title, _ = draw_result(image, result, TASK)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    axes[0].imshow(image); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(vis); axes[1].set_title(title); axes[1].axis("off")
plt.tight_layout(); plt.show()

---
## Bonus — Run ALL tasks (detect + ocr + layout + table)

Same as: `--task detect --task ocr --task layout --task table`

Running all four tasks on one page exercises the full dispatch table. Total compute scales linearly in the number of tasks because each call runs a separate forward–generate pass:

$$
T_{\text{total}} \approx \sum_{t \in \{\text{detect, ocr, layout, table}\}} T_{\text{gen}}(t)
$$

Use this to benchmark end-to-end document understanding latency before quantization. In notebook 02, **SmoothQuant** (Stage 4) is the best starting point when vision activations have outliers.

In [ ]:
if RUN_ALL_TASKS:
    all_results = {}
    for t in ["detect", "ocr", "layout", "table"]:
        print(f"Running {t}...")
        if BACKEND == "florence2":
            all_results[t] = run_florence_task(t, image, processor, model, DEVICE)
        else:
            all_results[t] = run_qwen_task(t, image, processor, model, MAX_NEW_TOKENS)
    print("\n" + "="*60)
    for t, res in all_results.items():
        print(f"\n--- {t.upper()} ---")
        if t == "detect":
            print(f"  {len(res.get('text_lines', []))} text lines")
        elif t == "ocr":
            print(f"  {len(res.get('text', ''))} chars of text")
        elif t == "layout":
            print(f"  {len(res.get('blocks', []))} layout blocks")
        else:
            print(f"  tables: {len(res.get('tables', []))}  note: {res.get('note', '')}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, t in zip(axes.flat, ["detect", "ocr", "layout", "table"]):
        if t == "ocr":
            ax.imshow(image); ax.set_title(f"ocr: {len(all_results[t].get('text',''))} chars"); ax.axis("off")
        elif t in ("detect", "layout", "table"):
            vis, title, _ = draw_result(image, all_results[t], t)
            ax.imshow(vis); ax.set_title(title); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Set RUN_ALL_TASKS = True to run all four tasks.")

---
## Bridge to Part 2 — SmoothQuant preview

> **Naming note:** **SmoothQuant** (Xiao et al., ICML 2023) is often mis-typed as *“Spin Quant”*. It is **not** **SpinQuant** (Liu et al., ICLR 2025) — that method uses *learned rotation matrices* and is listed separately below.

After you run OCR in this notebook, [02 — Quantization](02_ocr_pipeline_quant.ipynb) shrinks $f_\theta$ for mobile. The method most relevant to **activation outliers** in vision–language models is **SmoothQuant**.

### The problem SmoothQuant solves

Activations $\mathbf{X}$ have heavy-tailed channels: $\max |X_j| \gg \mathrm{median}(|X_j|)$. Uniform int8 scaling wastes range on outliers → large quantization error in matmul $\mathbf{Y} = \mathbf{X}\mathbf{W}^\top$.

### Migration formula (mathematically equivalent transform)

For each input channel $j$ and hyperparameter $\alpha \in [0,1]$:

$$
s_j = \frac{\max_m |X_j^{(m)}|^\alpha}{\max |W_j|^{1-\alpha}}, \qquad
W'_j = \frac{W_j}{s_j}, \qquad X'_j = X_j \cdot s_j
$$

### Theorem (output preserved in exact arithmetic)

$$
\sum_j X_j W_j = \sum_j X'_j W'_j = \sum_j (X_j s_j)(W_j / s_j)
$$

**Proof:** elementwise product unchanged; therefore each row of $\mathbf{X}\mathbf{W}^\top$ equals $\mathbf{X}'(\mathbf{W}')^\top$. SmoothQuant then quantizes the **easier** smoothed weights $\mathbf{W}'$ and applies $s_j$ to activations at runtime (folded into preceding op in deployment).

### Effect of $\alpha$

| $\alpha$ | Activation range after smooth | Weight range after smooth |
|----------|------------------------------|---------------------------|
| $0$ | unchanged ($A_j$) | unchanged ($\|W_j\|_\infty$) |
| $0.5$ | balanced (paper default) | balanced |
| $1$ | fully migrated to weights | minimal |

Implemented from scratch in **[02 Stage 4 — SmoothQuant](02_ocr_pipeline_quant.ipynb)** with alpha sweep demo (Stage 4b) and full 5-method OCR compare (Stage 8).

### SpinQuant (latest, rotation-based — separate method)

**SpinQuant** applies orthogonal rotations before quantize:

$$
\mathbf{Y} = \mathbf{X}\mathbf{W}^\top = (\mathbf{X}\mathbf{R})(\mathbf{R}^\top\mathbf{W}^\top), \quad \mathbf{R}^\top\mathbf{R} = \mathbf{I}
$$

Implemented in **[02 Stage 4c — SpinQuant](02_ocr_pipeline_quant.ipynb)** with Givens learning (Stage 4d demo).

### ConvRot (group-wise RHT — plug-and-play W4A4)

Partition channels into blocks of size $N_0$ (default 256). Apply **regular Hadamard** per block:

$$
\mathbf{H}_{4^{k+1}} = \mathbf{H}_{4^k} \otimes \mathbf{H}_4, \quad
\mathbf{Y} = \sum_i \text{RHT}(\mathbf{X}_i)\,\text{RHT}(\mathbf{W}_i)^\top
$$

**Proof (orthogonality):** Kronecker product of orthogonal matrices is orthogonal ⇒ $\text{RHT}(\mathbf{X}_i)\,\text{RHT}(\mathbf{W}_i)^\top = \mathbf{X}_i \mathbf{W}_i^\top$ per block.

**vs SpinQuant:** ConvRot uses fixed RHT ($O(K)$); SpinQuant learns Givens ($O(K \cdot \text{\#rotations})$). ConvRot excels at row-wise outliers in VLMs/DiTs.

Implemented in **[02 Stage 4e — ConvRot](02_ocr_pipeline_quant.ipynb)** with `CONVROT_GROUP_SIZE` ∈ {16, 64, 256, 1024}.

---
## Full pipeline map (all functions)

End-to-end data flow from PIL image to JSON:

```
run_task(image, task)                    ← entry point
  └─ _dispatch → _handlers()[task]
       └─ _generate(image, task_prompt)
            A. pad_info(image)           → padded, pad_x, pad_y, orig_w, orig_h
            B. processor(text, images)   → input_ids, pixel_values
            C. model.generate(...)       → generated_ids
            D. batch_decode(...)         → raw_text
       └─ _parse_florence(...)
            post_process_generation(...) → parsed dict
       └─ quad_to_bbox / unmap_bbox     → original image coords
       └─ clean_florence_text           → clean strings
       └─ task_result(...)              → final JSON
```

**Composition view:** $\text{JSON} = \mathcal{H}_t \circ \mathcal{P}_t \circ \mathcal{D} \circ \mathcal{G} \circ \mathcal{T}(\mathbf{I}, t)$ where $\mathcal{T}$ tokenizes, $\mathcal{G}$ generates, $\mathcal{D}$ decodes, $\mathcal{P}_t$ parses, and $\mathcal{H}_t$ is the task handler.

CLI equivalent:
```bash
python -m vlm_pipeline assets/table_page.png \
  --backend florence2 --task detect --task ocr --page 0
```

**Next:** [02 — Quantization (GPTQ, AWQ, SmoothQuant, SpinQuant, ConvRot)](02_ocr_pipeline_quant.ipynb) — see preview above; notebook 02 implements all five from scratch.